# 10 Full Lyrics Similarity

The previous notebook incorporated engineered lyrics features describing structural properties of song lyrics.

While these features provided additional information, they did not capture the semantic content of the lyrics themselves.

This notebook therefore explores a richer representation based on the complete lyrics text available in the tracks dataset.

The objective is to evaluate whether semantic information extracted directly from lyrics improves similarity-based recommendations compared to the previous models.

In [1]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / "data" / "processed"

processed_dir

WindowsPath('c:/Users/hp/OneDrive - WU Wien/Desktop/Denis_Masters/Semester 2/Solution Engineering - Python/Spotify_Project/spotify_project/data/processed')

In [2]:
tracks = pd.read_parquet(processed_dir / "tracks_clean.parquet")
audio_features = pd.read_parquet(processed_dir / "audio_features_clean.parquet")

model_a_features = pd.read_csv(
    processed_dir / "model_a_audio_features.csv"
)["feature"].tolist()

print(f"Tracks: {len(tracks):,}")
print(f"Audio Features: {len(audio_features):,}")
print(f"Model A Audio Features: {len(model_a_features):,}")

Tracks: 95,977
Audio Features: 95,948
Model A Audio Features: 206


# 1. Lyrics Text Audit

Before using the full lyrics text, the lyrics column must be checked for placeholder values.

Earlier inspection showed that some rows contain values such as "-99", empty text, "Instrumental", or unreleased-lyrics messages.

These rows are removed before building the full-lyrics similarity model.

In [3]:
placeholder_mask = (
    tracks["lyrics"].eq("-99")
    |
    tracks["lyrics"].str.contains(
        "Lyrics for this song have yet to be released",
        case=False,
        na=False
    )
    |
    tracks["lyrics"].str.contains(
        "Instrumental",
        case=False,
        na=False
    )
    |
    (tracks["lyrics"].str.strip() == "")
)

lyrics_audit = pd.DataFrame({
    "Metric": [
        "Total Tracks",
        "Valid Lyrics",
        "Placeholder Lyrics",
        "Valid Lyrics (%)",
        "Placeholder Lyrics (%)"
    ],
    "Value": [
        len(tracks),
        (~placeholder_mask).sum(),
        placeholder_mask.sum(),
        round((~placeholder_mask).mean() * 100, 2),
        round(placeholder_mask.mean() * 100, 2)
    ]
})

lyrics_audit

,Metric,Value
0,Total Tracks,95977.00
1,Valid Lyrics,79430.00
2,Placeholder Lyrics,16547.00
3,Valid Lyrics (%),82.76
4,Placeholder Lyrics (%),17.24


In [4]:
tracks_with_valid_lyrics = tracks[
    ~placeholder_mask
].copy()

tracks_with_valid_lyrics = tracks_with_valid_lyrics[
    tracks_with_valid_lyrics["id"].isin(audio_features["track_id"])
].copy()

print(f"Tracks with valid lyrics and audio features: {len(tracks_with_valid_lyrics):,}")

Tracks with valid lyrics and audio features: 79,405


# 2. Building the Model C Dataset

Model C combines audio features with the full lyrics text.

The audio features provide musical and acoustic similarity, while the lyrics text provides semantic information.

Only tracks with both valid lyrics and audio features are used.

In [5]:
model_c_master = tracks_with_valid_lyrics.merge(
    audio_features,
    left_on="id",
    right_on="track_id",
    how="inner"
)

model_c_master.shape

(79405, 238)

In [6]:
model_c_master[
    [
        "name",
        "popularity",
        "danceability",
        "energy",
        "valence",
        "tempo",
        "lyrics"
    ]
].head()

,name,popularity,danceability,energy,valence,tempo,lyrics
0,Blood,28.0,0.698,0.606,0.6220,115.018,\r\n\r\nPerhaps I am bound to be restless\r\nA...
1,Already Gone,45.0,0.367,0.349,0.1920,81.850,\r\nIf there were an ocean\r\nWe’d be wading i...
2,Creature Kind,47.0,0.748,0.666,0.3590,114.982,\r\n\r\nPictures in my mind of a world that I ...
3,Greatest Comedian,36.0,0.801,0.610,0.9420,104.199,"\r\n\r\nYou're the beauty, the stranger in the..."
4,I Wish I Was A Shark,32.0,0.515,0.351,0.0383,139.926,\r\n\r\nYou leapt from crumbling bridges watch...


# 3. Preparing Audio Features

The same audio feature list from Model A is reused to keep the comparison consistent.

This ensures that differences between Model A and Model C are caused by the added lyrics representation rather than changes in the audio feature space.

In [7]:
X_audio_c = model_c_master[
    model_a_features
].copy()

print(f"Audio feature matrix shape: {X_audio_c.shape}")
print(f"Missing values: {X_audio_c.isna().sum().sum()}")

Audio feature matrix shape: (79405, 206)
Missing values: 0


# 4. Lyrics Vectorization

The full lyrics text cannot be used directly in similarity calculations.

Therefore, the lyrics are transformed into a numerical representation using TF-IDF (Term Frequency–Inverse Document Frequency).

TF-IDF assigns higher importance to words that are distinctive for a song while reducing the influence of very common words.

This representation allows lyrical similarity to be incorporated into the recommendation process.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    min_df=5,
    max_df=0.8
)

X_lyrics_tfidf = tfidf.fit_transform(
    model_c_master["lyrics"]
)

print(
    f"TF-IDF Matrix Shape: "
    f"{X_lyrics_tfidf.shape}"
)

print(
    f"Vocabulary Size: "
    f"{len(tfidf.get_feature_names_out()):,}"
)

TF-IDF Matrix Shape: (79405, 5000)
Vocabulary Size: 5,000


### Inspect Vocabulary

In [9]:
tfidf_terms = pd.DataFrame({
    "term": tfidf.get_feature_names_out()
})

tfidf_terms.sample(
    20,
    random_state=42
)

,term
1501,final
2586,meiner
2653,mile
1055,did
705,cigarette
106,alguém
589,candle
2468,lu
2413,lleva
1600,frozen


### Most Frequent TF-IDF Terms

In [10]:
term_importance = np.asarray(
    X_lyrics_tfidf.sum(axis=0)
).flatten()

top_terms = pd.DataFrame({
    "term": tfidf.get_feature_names_out(),
    "importance": term_importance
})

top_terms = top_terms.sort_values(
    "importance",
    ascending=False
)

top_terms.head(20)

,term,importance
3401,que,3674.367985
2459,love,2439.986833
2268,la,2415.925579
2976,oh,2380.631755
1130,don,2318.609569
2240,know,2203.295734
2371,like,2035.090073
2173,just,1852.898250
4806,yeah,1649.938957
1252,el,1626.536449


### TF-IDF Vocabulary Observation

The TF-IDF matrix was successfully created with 5,000 lyric terms.

However, the vocabulary inspection shows that the dataset contains lyrics in multiple languages. Since only English stop words were removed, several common non-English words remain highly ranked.

This means that additional stop-word filtering is needed before using the TF-IDF representation for recommendations.

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{2,}\b",
    max_df=1.0,
    min_df=5
)

X_counts = count_vectorizer.fit_transform(
    model_c_master["lyrics"]
)

terms = count_vectorizer.get_feature_names_out()

term_document_frequency = np.asarray(
    (X_counts > 0).sum(axis=0)
).flatten()

term_frequency = np.asarray(
    X_counts.sum(axis=0)
).flatten()

term_stats = pd.DataFrame({
    "term": terms,
    "document_frequency": term_document_frequency,
    "term_frequency": term_frequency,
    "document_share": term_document_frequency / len(model_c_master)
})

term_stats = term_stats.sort_values(
    "document_share",
    ascending=False
)

term_stats.head(50)

,term,document_frequency,term_frequency,document_share
46983,me,53487,371559,0.673597
75637,the,45988,593137,0.579157
76443,to,44082,322357,0.555154
84934,you,43743,611498,0.550885
2948,and,42073,316166,0.529853
35847,in,40371,214353,0.508419
37268,it,39343,295448,0.495473
52004,no,36547,199209,0.460261
50137,my,35772,223317,0.450501
53541,on,34161,163857,0.430212


In [15]:
data_driven_stop_words = (
    term_stats[
        term_stats["document_share"] > 0.20
    ]["term"]
    .tolist()
)

data_driven_stop_words[:50]

['me',
 'the',
 'to',
 'you',
 'and',
 'in',
 'it',
 'no',
 'my',
 'on',
 'that',
 'of',
 'all',
 'your',
 'for',
 'is',
 'so',
 'but',
 'can',
 'be',
 'we',
 'know',
 'don',
 'like',
 'with',
 'just',
 'when',
 'up',
 'this',
 're',
 'what',
 'de',
 'do',
 'love',
 'oh',
 'now',
 'out',
 'que',
 'if',
 'll',
 'go',
 'got',
 'time',
 'see',
 'la',
 'en',
 'get',
 'yeah',
 'one',
 've']

In [16]:
tfidf = TfidfVectorizer(
    stop_words=data_driven_stop_words,
    max_features=5000,
    min_df=5,
    max_df=0.8,
    token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{2,}\b"
)

X_lyrics_tfidf = tfidf.fit_transform(
    model_c_master["lyrics"]
)

print(f"TF-IDF Matrix Shape: {X_lyrics_tfidf.shape}")
print(f"Vocabulary Size: {len(tfidf.get_feature_names_out()):,}")

TF-IDF Matrix Shape: (79405, 5000)
Vocabulary Size: 5,000


In [17]:
term_importance = np.asarray(
    X_lyrics_tfidf.sum(axis=0)
).flatten()

top_terms = pd.DataFrame({
    "term": tfidf.get_feature_names_out(),
    "importance": term_importance
}).sort_values(
    "importance",
    ascending=False
)

top_terms.head(30)

,term,importance
1272,el,1740.874835
4275,te,1634.876873
4497,tu,1379.357753
275,baby,1327.925256
2379,let,1327.830277
2676,mi,1305.206055
3873,she,1178.844953
200,are,1172.684884
4339,they,1153.171818
4764,want,1152.750688


In [11]:
custom_stop_words = list(
    set([
        # English / lyric filler
        "oh", "yeah", "yea", "hey", "ah", "uh", "ooh", "la", "na",
        "don", "ll", "ve", "im", "youre", "wanna", "gonna",

        # Spanish / Portuguese common words
        "que", "de", "el", "la", "los", "las", "en", "un", "una",
        "te", "tu", "yo", "mi", "me", "se", "es", "por", "para",
        "con", "no", "si", "ya", "lo", "al", "del",

        # German common words
        "ich", "du", "der", "die", "das", "und", "ist", "nicht",
        "ein", "eine", "im", "in", "den", "dem", "zu", "mit",

        # French common words
        "je", "tu", "il", "elle", "le", "les", "des", "un", "une",
        "et", "est", "pas", "dans", "pour", "avec"
    ])
)

In [12]:
tfidf = TfidfVectorizer(
    stop_words=custom_stop_words,
    max_features=5000,
    min_df=5,
    max_df=0.8
)

X_lyrics_tfidf = tfidf.fit_transform(
    model_c_master["lyrics"]
)

print(f"TF-IDF Matrix Shape: {X_lyrics_tfidf.shape}")
print(f"Vocabulary Size: {len(tfidf.get_feature_names_out()):,}")

TF-IDF Matrix Shape: (79405, 5000)
Vocabulary Size: 5,000


In [13]:
term_importance = np.asarray(
    X_lyrics_tfidf.sum(axis=0)
).flatten()

top_terms = pd.DataFrame({
    "term": tfidf.get_feature_names_out(),
    "importance": term_importance
}).sort_values(
    "importance",
    ascending=False
)

top_terms.head(20)

,term,importance
4825,you,7678.546154
4220,the,6231.110615
168,and,3831.199658
4286,to,3805.423355
2097,it,3675.939424
2771,my,3164.937444
4660,we,2413.956126
2989,on,2267.982082
4218,that,2240.172357
4828,your,2205.355557


### Vocabulary Analysis

Inspection of the lyrics corpus revealed that the dataset contains songs in multiple languages, including English, Spanish, and Portuguese.

A data-driven stop-word approach was therefore adopted. Terms appearing in more than 20% of tracks were automatically identified and removed before TF-IDF vectorization.

This procedure removed highly frequent low-information words such as "the", "you", "me", "que", and "la", allowing the representation to focus more strongly on content-bearing terms.

The resulting vocabulary contains words that are more indicative of lyrical themes and song content, making it more suitable for similarity-based recommendation.

# 5. Combining Audio and Lyrics Representations

Model C combines two complementary sources of information:

- Audio features describing musical characteristics
- TF-IDF features describing lyrical content

The resulting representation captures both acoustic similarity and semantic similarity between songs.

In [18]:
from sklearn.preprocessing import StandardScaler

scaler_c = StandardScaler()

X_audio_c_scaled = scaler_c.fit_transform(
    X_audio_c
)

print(
    f"Scaled Audio Matrix: "
    f"{X_audio_c_scaled.shape}"
)

Scaled Audio Matrix: (79405, 206)


In [19]:
from scipy.sparse import hstack
from scipy.sparse import csr_matrix

X_audio_sparse = csr_matrix(
    X_audio_c_scaled
)

X_model_c = hstack([
    X_audio_sparse,
    X_lyrics_tfidf
])

In [20]:
print(
    f"Combined Feature Matrix: "
    f"{X_model_c.shape}"
)

Combined Feature Matrix: (79405, 5206)


In [21]:
model_c_summary = pd.DataFrame({
    "Metric": [
        "Tracks",
        "Audio Features",
        "Lyrics Features",
        "Total Features"
    ],
    "Value": [
        len(model_c_master),
        X_audio_c.shape[1],
        X_lyrics_tfidf.shape[1],
        X_model_c.shape[1]
    ]
})

model_c_summary

,Metric,Value
0,Tracks,79405
1,Audio Features,206
2,Lyrics Features,5000
3,Total Features,5206


# 6. Similarity Computation

The combined feature representation is used to compute similarity between tracks.

Cosine similarity is again employed to identify the nearest neighbours in the combined audio-lyrics feature space.

In [22]:
from sklearn.neighbors import NearestNeighbors

nn_model_c = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11
)

nn_model_c.fit(X_model_c)

print("Model C fitted successfully.")

Model C fitted successfully.


# 7. Generating Model C Recommendations

Model C retrieves similar tracks using the combined audio and full-lyrics feature space.

Compared to Models A and B, this model can consider both musical similarity and lyrical content similarity.

In [23]:
def recommend_full_lyrics_tracks(
    song_name,
    n_recommendations=10,
    remove_same_title=True
):

    matches = model_c_master[
        model_c_master["name"].str.contains(
            song_name,
            case=False,
            na=False
        )
    ]

    if matches.empty:
        print(f"No song found for: {song_name}")
        return None

    query_idx = matches["popularity"].idxmax()
    query_title = model_c_master.loc[query_idx, "name"]

    distances, indices = nn_model_c.kneighbors(
        X_model_c[query_idx],
        n_neighbors=n_recommendations + 20
    )

    recommendations = model_c_master.iloc[indices[0]].copy()
    recommendations["cosine_distance"] = distances[0]

    recommendations = recommendations[
        recommendations.index != query_idx
    ]

    if remove_same_title:
        recommendations = recommendations[
            recommendations["name"].str.lower()
            != query_title.lower()
        ]

    result = recommendations[
        [
            "name",
            "popularity",
            "danceability",
            "energy",
            "valence",
            "tempo",
            "cosine_distance"
        ]
    ].head(n_recommendations)

    print(f"Recommendations for: {query_title}")

    return result

In [24]:
recommend_full_lyrics_tracks("Billie Jean")

Recommendations for: Billie Jean


,name,popularity,danceability,energy,valence,tempo,cosine_distance
53477,Michael Jackson x Mark Ronson: Diamonds are In...,59.0,0.672,0.832,0.549,117.298,0.254280
45173,Hooligan,36.0,0.612,0.898,0.782,150.030,0.261201
79360,Pressure - Blanke Remix,44.0,0.512,0.944,0.280,187.916,0.267482
28069,Need Somebody,39.0,0.621,0.990,0.172,150.032,0.267940
79364,Hate Being Alone,55.0,0.501,0.969,0.170,74.767,0.268729
46780,All I Need (with Gucci Mane),59.0,0.492,0.801,0.329,150.036,0.269006
12844,Never Been In Love (feat. Icona Pop),47.0,0.631,0.929,0.695,121.080,0.270118
52444,Camo Diamond Rollie,52.0,0.604,0.984,0.139,145.095,0.272052
66740,Wild Thoughts - Dave Audé Dance Remix,53.0,0.698,0.924,0.641,116.040,0.273814
33712,Che Ne Sanno I 2000 (feat. Danti),54.0,0.744,0.982,0.333,128.033,0.275511


In [28]:
recommend_full_lyrics_tracks("Starboy")

Recommendations for: Starboy - Acoustic


,name,popularity,danceability,energy,valence,tempo,cosine_distance
31013,With U 2,35.0,0.680,0.2890,0.6760,78.528,0.263932
32827,You Are Not Alone,43.0,0.753,0.2080,0.2810,115.009,0.283419
10028,Next to You,50.0,0.443,0.3320,0.0689,87.839,0.307829
78926,Here Today,34.0,0.533,0.1160,0.0391,75.041,0.308540
19052,Touch Me,40.0,0.944,0.3820,0.5020,105.043,0.313029
46899,Loco,60.0,0.746,0.3810,0.2470,100.049,0.317311
21600,Génie,53.0,0.548,0.4330,0.1530,75.485,0.317449
5797,leave me behind,32.0,0.515,0.0402,0.0830,90.779,0.320933
14279,Let Me Go,46.0,0.563,0.4230,0.0839,92.882,0.322933
61958,You Let Me Walk Alone,57.0,0.451,0.2790,0.4240,74.786,0.327891


In [26]:
recommend_full_lyrics_tracks("Sign of the Times")

Recommendations for: Sign of the Times


,name,popularity,danceability,energy,valence,tempo,cosine_distance
68648,It Takes A Fool To Remain Sane,56.0,0.692,0.585,0.386,122.955,0.383648
50391,Un Amor Como El Nuestro,37.0,0.616,0.274,0.646,175.954,0.469777
26268,Better In The Dark,53.0,0.670,0.758,0.492,110.024,0.469986
49193,Honest,44.0,0.419,0.541,0.173,147.416,0.476062
15702,Sensations,51.0,0.439,0.832,0.272,143.993,0.488633
72877,Just Goes to Show,34.0,0.493,0.807,0.408,100.246,0.490344
55206,Not Too Late,55.0,0.657,0.693,0.271,120.041,0.494470
29269,Te Quiero,46.0,0.549,0.449,0.378,81.983,0.497711
42367,Not Too Late,49.0,0.657,0.693,0.271,120.041,0.501060
59237,Hero,66.0,0.341,0.539,0.267,174.564,0.505992


In [29]:
recommend_full_lyrics_tracks("Like a Prayer")

Recommendations for: Like a Prayer


,name,popularity,danceability,energy,valence,tempo,cosine_distance
45467,Sk8er Boi,76.0,0.487,0.900,0.4840,149.937,0.273495
31585,The Resistance,69.0,0.483,0.941,0.5030,156.033,0.287424
76483,All Out Life,74.0,0.499,0.970,0.0506,106.558,0.287680
12427,Still Into You,73.0,0.602,0.923,0.7650,136.010,0.299442
6516,SING,61.0,0.606,0.942,0.3730,110.980,0.303995
69588,Psychosocial,73.0,0.576,0.989,0.3520,135.093,0.324447
2530,Danky,36.0,0.647,0.932,0.3660,107.973,0.328175
60171,Be Alright,21.0,0.451,0.796,0.4920,94.986,0.328582
20021,Psychosocial,52.0,0.576,0.989,0.3520,135.093,0.332579
59651,Hungry Are the Damned,32.0,0.132,0.994,0.1600,144.887,0.332824


### First Model C Observation

Model C successfully combines audio features with TF-IDF lyrics features.

However, the recommendation outputs remain very similar to the previous audio-based models. This suggests that the audio feature space may still dominate the similarity calculation, while the lyrics representation has only limited influence in the combined matrix.

To test this, an additional weighted version of Model C is created where the TF-IDF lyrics representation receives more influence.

A weighting experiment was added to test whether the TF-IDF lyrics representation affects the recommendations when given stronger influence.

The value 3.0 is used as an initial exploratory weight, not as a tuned parameter. It simply increases the contribution of the lyrics block relative to the audio block.

In [30]:
lyrics_weight = 3.0

X_model_c_weighted = hstack([
    X_audio_sparse,
    X_lyrics_tfidf * lyrics_weight
])

print(f"Weighted Model C Matrix: {X_model_c_weighted.shape}")

Weighted Model C Matrix: (79405, 5206)


In [31]:
nn_model_c_weighted = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11
)

nn_model_c_weighted.fit(X_model_c_weighted)

print("Weighted Model C fitted successfully.")

Weighted Model C fitted successfully.


In [32]:
def recommend_full_lyrics_weighted(
    song_name,
    n_recommendations=10,
    remove_same_title=True
):

    matches = model_c_master[
        model_c_master["name"].str.contains(
            song_name,
            case=False,
            na=False
        )
    ]

    if matches.empty:
        print(f"No song found for: {song_name}")
        return None

    query_idx = matches["popularity"].idxmax()
    query_title = model_c_master.loc[query_idx, "name"]

    distances, indices = nn_model_c_weighted.kneighbors(
        X_model_c_weighted[query_idx],
        n_neighbors=n_recommendations + 20
    )

    recommendations = model_c_master.iloc[indices[0]].copy()
    recommendations["cosine_distance"] = distances[0]

    recommendations = recommendations[
        recommendations.index != query_idx
    ]

    if remove_same_title:
        recommendations = recommendations[
            recommendations["name"].str.lower()
            != query_title.lower()
        ]

    result = recommendations[
        [
            "name",
            "popularity",
            "danceability",
            "energy",
            "valence",
            "tempo",
            "cosine_distance"
        ]
    ].head(n_recommendations)

    print(f"Recommendations for: {query_title}")

    return result

In [33]:
recommend_full_lyrics_weighted("Billie Jean")

Recommendations for: Billie Jean


,name,popularity,danceability,energy,valence,tempo,cosine_distance
53477,Michael Jackson x Mark Ronson: Diamonds are In...,59.0,0.672,0.832,0.549,117.298,0.270879
45173,Hooligan,36.0,0.612,0.898,0.782,150.030,0.278754
79360,Pressure - Blanke Remix,44.0,0.512,0.944,0.280,187.916,0.284341
79364,Hate Being Alone,55.0,0.501,0.969,0.170,74.767,0.284682
28069,Need Somebody,39.0,0.621,0.990,0.172,150.032,0.285699
52444,Camo Diamond Rollie,52.0,0.604,0.984,0.139,145.095,0.287980
46780,All I Need (with Gucci Mane),59.0,0.492,0.801,0.329,150.036,0.291096
33712,Che Ne Sanno I 2000 (feat. Danti),54.0,0.744,0.982,0.333,128.033,0.291447
66740,Wild Thoughts - Dave Audé Dance Remix,53.0,0.698,0.924,0.641,116.040,0.292263
57749,Stay (feat. Holly),47.0,0.422,0.947,0.213,169.983,0.292710


In [34]:
recommend_full_lyrics_weighted("Starboy")

Recommendations for: Starboy - Acoustic


,name,popularity,danceability,energy,valence,tempo,cosine_distance
31013,With U 2,35.0,0.680,0.2890,0.6760,78.528,0.302057
32827,You Are Not Alone,43.0,0.753,0.2080,0.2810,115.009,0.317323
78926,Here Today,34.0,0.533,0.1160,0.0391,75.041,0.339024
10028,Next to You,50.0,0.443,0.3320,0.0689,87.839,0.339778
19052,Touch Me,40.0,0.944,0.3820,0.5020,105.043,0.341622
5797,leave me behind,32.0,0.515,0.0402,0.0830,90.779,0.343910
46899,Loco,60.0,0.746,0.3810,0.2470,100.049,0.346064
21600,Génie,53.0,0.548,0.4330,0.1530,75.485,0.349757
54464,En attendant demain - Single,24.0,0.602,0.3340,0.2330,81.809,0.358654
14279,Let Me Go,46.0,0.563,0.4230,0.0839,92.882,0.359919


Increasing the influence of the lyrics representation slightly increased cosine distances, but did not substantially change the top recommendations. This suggests that the current TF-IDF lyrics representation has limited effect when combined directly with the audio feature space.